# Layer 4: Field Planning (class notes)

**What this layer is, in one line:** it decides where to send the field officers so that the village calls become right as often as possible.

These notes explain the ideas. You do not need to know how the code is written. **Everything in this layer is a computer simulation.** There was no real field campaign.

## 1. The goal

Layer 3 gives every village a yes/no call and a measure of how sure it is. Some calls are shaky. But we have only **3 officers, 4 hours a day**, and some roads are flooded. Layer 4 answers three questions:

| Question | Short answer |
|---|---|
| Which check should come next? | The one most likely to turn a wrong call into a right one |
| Which villages and roads today? | The best payoff per extra minute of driving, on a map that knows about flooded roads |
| How do we keep the sample fair? | Visit a few villages picked at random with known chances |

## 2. Which check next?

**Chance a call is wrong.** A village's estimate comes with an uncertainty. If its estimate is very close to the 25 percent line, the call could easily be wrong. If it is far from the line, the call is almost surely right, even if the estimate is rough.

**The rule.** Send the next check where it is expected to cut the number of wrong calls the most. To judge this, we imagine what the check might find: it would move the estimate a little and make it more certain, and we work out the expected drop in the chance of being wrong.

**Why this is smart.** The usual alternative is "check where we are least certain". That wastes checks on villages that are clearly far above or below the line. Our rule ignores them and focuses on villages near the line, where one more check can actually change the answer.

**Does it work?** In simulation with 80 villages, 60 test scenarios, at **16 checks per village**:

| Rule | Wrong calls |
|---|---|
| Random placement | 14.0 percent |
| Proportional to village size | 13.8 percent |
| Check the most uncertain | 12.6 percent |
| **Our rule** | **8.6 percent** |

We reach the others' 32-check accuracy with about 12 checks.

**Honest limits.**
* With only **2 to 4 checks per village**, every rule is **worse than just using the map**. Early checks are noisy and swing the estimate. The map alone gets 11.4 percent wrong; ours gets 19.2 at 2 checks and 14.7 at 4.
* On real map errors (91 test pseudo-villages) our rule is best up to 16 checks but **loses its edge at 32** (3.6 percent against 3.1 for "check the most uncertain").

## 3. Roads: the flood-aware map

The road map from Layer 1 has a normal travel time for each road piece. Layer 2 gives the chance each road piece is flooded. We make a **flood-aware** map: roads that are probably flooded get slower, and roads that are very probably flooded (above 70 percent) are treated as closed. Then a shortest-path method (Dijkstra, from Layer 1) finds the fastest way to every village.

In our simulated truth a flooded road takes **8 times** as long. Surprisingly, planning with 8 times is **worse** than ignoring floods. The reason: on the day, an officer picks whichever route happens to be dry, so the average slowdown overstates the real one. So the planning slowdown was **fitted** instead, giving **4 times**. Travel-time error of the plan against the simulated truth:

| Planning map | Typical error |
|---|---|
| Ignore floods | 29.5 percent (too optimistic) |
| Use the true factor 8 | 52.1 percent (too pessimistic) |
| **Fitted factor 4** | **21.0 percent** |

## 4. Which villages and roads today?

Each morning the planner works out, for every village, how much a visit (12 checks, about 18 minutes) would reduce wrong calls. Then it repeatedly adds the village with the best **gain per extra minute** to an officer's route, as long as the officer's day still fits in 4 hours. This kind of problem is called **orienteering**: collect the most reward in a time limit. It is solved by a quick, greedy rule, which is good but not perfect.

Officers then drive the real (simulated) roads and turn back if time runs out. Next morning everything is recomputed with the new checks.

**Compared with simple plans over 5 days, in simulation (30 villages, 60 scenarios):**

| Plan | Wrong calls after day 5 |
|---|---|
| Nearest village first (the usual practice) | 13.3 percent |
| Highest gain, ignore travel | 11.1 percent |
| Gain per minute, ignoring floods | 10.6 percent |
| **Gain per minute, flood-aware** | **9.7 percent** |
| Map alone (no visits) | 10.9 percent |

Flood-aware beats nearest-first by about **3.6 points**. Honest points: flood-aware against flood-blind is not a statistically clear win in the simulation (it does about 27 percent less driving for the same result), and on days 1 to 3 every plan is worse than the map alone.

## 5. Keeping the sample fair: the random "spine"

Routes naturally favour villages near the depot and on dry roads. Those checks are not a fair sample of the whole region, so anything we average across villages (for example, how often the map is right) becomes biased.

The fix is a **spine**: before day 1, pick about **8 villages at random**, giving larger villages a higher chance (but at least 5 percent for all). Because we know each village's chance of being picked, we can weight checks to remove bias later (see Layer 1, section 8). These villages are always visited first, and 6 plots per group are checked inside them. The spine only draws from villages reachable within a reasonable part of a day, which covers about **half** of all plots; we say that plainly.

Result in simulation: weighted spine estimates showed **no detectable bias**, while pooling all routed checks over-stated one rate by about 1.5 points. But routing bias is small in our simulated world, so the spine buys a **guarantee**, not a smaller error. It also costs no measurable accuracy in village calls.

## 6. Fair testing

Few settings were chosen (how fast to plan floods, which roads to close, how to weight the spine). They were chosen on separate **tuning scenarios**. The results above come from other scenarios that were never used for tuning, so they are not flattered by the tuning.

**What is real and what is simulated.** The road map (OpenStreetMap) and the Layer 2 chip maps are real. The flood layer over the study block, the field truth, the officers and the 8 times slowdown are simulated or assumed.

## 7. What this layer hands to the tool

A day-1 plan: for each officer the villages in order, the road path and the planned minutes (our run: 3 officers with 8, 7 and 5 villages), plus a ranking of next checks and the chance each village was drawn for the spine. In the committed tool run, the routes shown are a stand-in fixture and no Layer 4 checks are used.

## 8. Key numbers

| Thing | Value |
|---|---|
| Officers, day, visit | **3** officers, **4 hours**, **12 checks** (about 18 minutes) per visit |
| Campaign length | **5 days** |
| Flooded road, truth vs plan | truth **8 times** slower, plan uses **4 times** |
| Close a road above | **70 percent** flood chance |
| Spine | about **8** villages, minimum chance **5 percent**, covers about **half** of plots |
| 16 checks, wrong calls | ours **8.6**, others 12.6 to 14.0, map alone 11.4 |
| 2 to 4 checks | every rule worse than the map |
| After 5 days | flood-aware plan **9.7**, nearest first **13.3** |

## 9. Glossary

* **Greedy**: always take the best-looking next step, without planning far ahead.
* **Orienteering problem**: collect the most reward within a time limit.
* **Flood-aware**: road times adjusted by the chance of flooding.
* **Spine**: a small random set of villages with known chances, always visited.
* **Tuning scenarios**: simulations used only to choose settings, never for scoring.
* **Gain**: expected drop in the chance of a wrong call.

## 10. Review questions

1. **What does Layer 4 do?** Decides where officers check and which roads they drive.
2. **How is the next check chosen?** Where it is expected to reduce wrong calls the most.
3. **Why not check the most uncertain village?** Villages far from the line stay right however uncertain; we focus on villages near it.
4. **How much better is it?** At 16 checks, 8.6 percent wrong against 12.6 to 14.0.
5. **Where does it not win?** With 2 to 4 checks every rule is worse than the map, and at 32 checks on real errors it loses its edge.
6. **Why does the plan assume 4 times, not 8?** Officers take whichever road is dry, so 8 overstates travel by about 50 percent.
7. **What is the spine and why?** A random set of villages with known chances, so checks can be weighted to remove routing bias.
8. **Does the spine cost accuracy?** No measurable loss.
9. **Is any of this real?** Roads and chip maps are real; the field work, flooding over the block and the slowdown are simulated or assumed.